# Step 09 — Score any arm against the human labels

**Input** — `06_validation_set.csv` plus whichever `*_predictions.parquet` files

exist (notebook 07 writes `rule`, notebook 08 writes `llm`)

**Output** — printed report; nothing is written.

Every arm present is scored through the same code, on the same rows, so the

numbers are comparable by construction.

## Two dimensions, two metrics

**Activities** are a multi-label *set*, so "wrong" has a direction.

Over-prediction misallocates zone capacity onto a building with no claim to it;

under-prediction removes the building from that activity's redistribution

entirely. These are different failures and are reported separately as

micro-averaged precision / recall, never collapsed into one accuracy number.

**Bosserhof class** is a single label per building, so plain accuracy is the

right metric — it either matches or it does not.

Micro-averaging is over label *instances*, not rows: a building that

over-predicts two activities contributes two false positives.

In [ ]:
import sys

sys.path.insert(0, str(__import__('pathlib').Path('..').resolve()))

import pandas as pd

from config import VALIDATION_SET_FILE, VALIDATION_DIR, arm_predictions

from validation_utils import (decode_final_validation_set, collapse_to_zone_activities,

                              resolve_prediction_bosserhof, score_activities, score_bosserhof,

                              per_activity_breakdown, format_activity_report,

                              format_bosserhof_report, wilson_interval)

from llm_utils import normalise_mid_labels

pd.set_option('display.width', 200)

val = decode_final_validation_set(pd.read_csv(VALIDATION_SET_FILE))

val['gml_id'] = val['gml_id'].astype(str)

print(f'{len(val):,} validated buildings')

print(f"  activities scoreable : {val['activities_truth'].notna().sum():,}")

print(f"  bosserhof scoreable  : {val['bosserhof_truth'].notna().sum():,}")

## 1. Load every arm that has been run

An arm's predictions are normalised the same way regardless of who produced them:

labels canonicalised, collapsed to the 7 zone activities, Bosserhof string put

through the same normaliser the truth side used. Comparing a semantically

normalised truth against a raw prediction would not be like-for-like.

In [ ]:
arms = {}

for name in ('rule', 'llm'):

    path = arm_predictions(name)

    if not path.exists():

        print(f'{name:5s} — not run yet, skipped')

        continue

    p = pd.read_parquet(path)

    p['gml_id'] = p['gml_id'].astype(str)

    p = p.drop_duplicates('gml_id', keep='last')

    if 'pred_zone_activities' not in p.columns:

        p['pred_zone_activities'] = p['mid_labels'].map(normalise_mid_labels).map(

            collapse_to_zone_activities)

    if 'pred_bosserhof' not in p.columns:

        p['pred_bosserhof'] = p['bosserhof_class'].map(resolve_prediction_bosserhof)

    m = val.merge(p[['gml_id', 'pred_zone_activities', 'pred_bosserhof']],

                  on='gml_id', how='left', validate='one_to_one')

    n_missing = m['pred_zone_activities'].isna().sum()

    assert n_missing == 0, f'{name}: {n_missing} validated rows have no prediction'

    arms[name] = m

    print(f'{name:5s} — {len(m):,} predictions loaded')

assert arms, 'no arm has been run yet — run notebook 07 and/or 08 first'

## 2. Report per arm

In [ ]:
summary = []

for name, m in arms.items():

    print('=' * 78)

    print(f'ARM: {name.upper()}')

    print('=' * 78)

    act = m[m['activities_truth'].notna()]

    am, _ = score_activities(list(zip(act['gml_id'], act['pred_zone_activities'],

                                      act['activities_truth'])))

    print(format_activity_report(am, 'ACTIVITIES'))

    bos = m[m['bosserhof_truth'].notna()]

    bm, _ = score_bosserhof(list(zip(bos['gml_id'], bos['pred_bosserhof'],

                                     bos['bosserhof_truth'])))

    lo, hi = wilson_interval(bm['n_correct'], bm['n_rows'])

    print()

    print(format_bosserhof_report(bm, 'BOSSERHOF'))

    print(f'  wrong                  : {bm["n_rows"] - bm["n_correct"]:,}')

    print(f'  95% CI                 : {lo:.1%} - {hi:.1%}')

    print()

    summary.append({'arm': name,

                    'act_buildings': am['n_rows'],

                    'act_precision': round(am['precision'], 4),

                    'act_recall': round(am['recall'], 4),

                    'act_exact_set': round(am['exact_match_rate'], 4),

                    'boss_buildings': bm['n_rows'],

                    'boss_correct': bm['n_correct'],

                    'boss_wrong': bm['n_rows'] - bm['n_correct'],

                    'boss_accuracy': round(bm['accuracy'], 4)})

## 3. Head to head

In [ ]:
print(pd.DataFrame(summary).to_string(index=False))

## 4. Which activities does each arm invent, and which does it overlook?

A single precision figure cannot show this, and the two have opposite

consequences for a transport model.

In [ ]:
for name, m in arms.items():

    act = m[m['activities_truth'].notna()]

    b = pd.DataFrame(per_activity_breakdown(

        list(zip(act['gml_id'], act['pred_zone_activities'], act['activities_truth']))))

    b['miss_rate'] = (b['missed'] / b['in_truth'].where(b['in_truth'] > 0)).round(3)

    print(f'\n--- {name} ---')

    print(b.sort_values('in_truth', ascending=False).to_string(index=False))

## 5. Where the score comes from

The workbook's pre-filled values are an earlier run of the LLM, so the *green*

rows are the ones that model already got right and the *red* rows are the ones it

got wrong. Splitting the score that way separates "reproduces what was already

correct" from "fixes what was wrong" — two very different abilities that a single

accuracy number hides.

In [ ]:
rows = []

for name, m in arms.items():

    bos = m[m['bosserhof_truth'].notna()].copy()

    bos['correct'] = [str(p or '').strip().lower() == str(t or '').strip().lower()

                      for p, t in zip(bos['pred_bosserhof'], bos['bosserhof_truth'])]

    for verdict, grp in bos.groupby('Bosserhof_class_mistakes_color'):

        rows.append({'arm': name,

                     'workbook_verdict': verdict,

                     'meaning': {'green': 'earlier model was RIGHT',

                                 'red': 'earlier model was WRONG'}.get(verdict, verdict),

                     'buildings': len(grp),

                     'correct': int(grp['correct'].sum()),

                     'accuracy': round(grp['correct'].mean(), 4)})

print(pd.DataFrame(rows).to_string(index=False))